I have a fixed budget of 1000 RON to purchase gifts for a group of 4 people (father, mother, boyfriend, sister). For each, I have identified a list of options consisting of 3 profile-specific gifts and 4 neutral(universal) gifts.

The goal is to **maximize the total satisfaction subject to the budget constraint.**


The algorithm must satisfy the following conditions:

1.   Economic Constraint: the total cost must be less than or equal to 1000 RON.
2.   Availability Constraint: the available stock limits for each product.
1.   Equity Constraints: each person must receive a minimum of 1 gift and a maximum of 4 gifts, to ensure a balanced distribution.



In [1]:
!pip install pulp

In [22]:
from pulp import *

def optimizare_cadouri():
    BUGET_TOTAL=1000.0  #RON

    #Destinatarii
    persoane=["Tata", "Mama", "Iubitul", "Sora"]

    cadouri_magazin={
        #Tata(Camping)
        "Cort":                    {"pret":249, "stoc":1},
        "Briceag multifunctional": {"pret":99,  "stoc":1},
        "Lanterna":                {"pret":75,  "stoc":2},

        #Mama(Flori)
        "Orhidee in ghiveci":      {"pret":49.99,"stoc":1},
        "Set unelte gradinarit":   {"pret":119.93,"stoc":1},
        "Vaza ceramica pictata":   {"pret":78.65,"stoc":1},

        #Prieten(Sport)
        "Smartwatch fitness":      {"pret":304.15,"stoc":1},
        "Rucsac sala":             {"pret":119,  "stoc":1},
        "Set benzi elastice":      {"pret":47.48,"stoc":2},

        #Sora(Make-up)
        "Mascara":                 {"pret":47, "stoc":1},
        "Paleta farduri":          {"pret":122,  "stoc":1},
        "Ruj":                     {"pret":42,  "stoc":2},

        #Pentru oricine daca mai raman bani sau trebuie completat numarul de cadouri
        "Baterie externa":         {"pret":116.99,"stoc":2},
        "Joc societate(monopoly)": {"pret":185.99,"stoc":1},
        "Termos inox":             {"pret":89, "stoc":3},
        "Ciocolata":               {"pret":13.99, "stoc":5}
    }

    lista_cadouri = list(cadouri_magazin.keys())

    preferinte = {
        "Cort":                    {"Tata": 10,  "Mama": 2,    "Iubitul": 6,    "Sora": 1},
        "Briceag multifunctional": {"Tata": 10,  "Mama": 3,    "Iubitul": 7,    "Sora": 1},
        "Lanterna":                {"Tata": 9,   "Mama": 4,    "Iubitul": 5,    "Sora": 2},

        "Orhidee in ghiveci":      {"Tata": 2,   "Mama": 10,   "Iubitul": 1,    "Sora": 6},
        "Set unelte gradinarit":   {"Tata": 5,   "Mama": 9,    "Iubitul": 1,    "Sora": 1},
        "Vaza ceramica pictata":   {"Tata": 1,   "Mama": 8,    "Iubitul": 0,    "Sora": 5},

        "Smartwatch fitness":      {"Tata": 5,   "Mama": 3,    "Iubitul": 10,   "Sora": 4},
        "Rucsac sala":             {"Tata": 3,   "Mama": 1,    "Iubitul": 9,    "Sora": 2},
        "Set benzi elastice":      {"Tata": 2,   "Mama": 2,    "Iubitul": 8,    "Sora": 2},

        "Mascara":                 {"Tata": 0,   "Mama": 6,    "Iubitul": 0,    "Sora": 10},
        "Paleta farduri":          {"Tata": 0,   "Mama": 5,    "Iubitul": 0,    "Sora": 9},
        "Ruj":                     {"Tata": 0,   "Mama": 5,    "Iubitul": 0,    "Sora": 8},


        "Baterie externa":         {"Tata": 8,   "Mama": 5,    "Iubitul": 8,    "Sora": 7},
        "Joc societate(monopoly)": {"Tata": 6,   "Mama": 6,    "Iubitul": 6,    "Sora": 6},
        "Termos inox":             {"Tata": 8,   "Mama": 4,    "Iubitul": 8,    "Sora": 4},
        "Ciocolata":               {"Tata": 7,   "Mama": 7,    "Iubitul": 3,    "Sora": 7}
    }

    prob = LpProblem("Optimizare_Cadouri_Familie", LpMaximize)
    x = LpVariable.dicts("Cumpara", (lista_cadouri, persoane), 0, 1, LpBinary)
    prob += lpSum([x[c][p] * preferinte[c][p] for c in lista_cadouri for p in persoane])

    #CONSTRÂNGERI:
    #1.Buget Total
    prob += lpSum([x[c][p] * cadouri_magazin[c]['pret'] for c in lista_cadouri for p in persoane]) <= BUGET_TOTAL, "Buget_Maxim"

    #2.Stoc
    for c in lista_cadouri:
        prob += lpSum([x[c][p] for p in persoane]) <= cadouri_magazin[c]['stoc'], f"Stoc_{c}"

    #3.Minim un cadou per persoană
    for p in persoane:
        prob += lpSum([x[c][p] for c in lista_cadouri]) >= 1, f"Minim_1_cadou_{p}"

    #4.Maxim patru cadouri per persoană
    for p in persoane:
        prob += lpSum([x[c][p] for c in lista_cadouri]) <= 4, f"Maxim_4_cadouri_{p}"

    prob.solve(PULP_CBC_CMD(msg=False))

    print(f"Status soluție: {LpStatus[prob.status]}")
    print(f"Fericire totală(puncte): {value(prob.objective)}")

    total_cheltuit = 0

    for p in persoane:
        print(f"\n Cadouri pentru {p.upper()}:")
        subtotal = 0
        satisfactie = 0
        for c in lista_cadouri:
            if x[c][p].varValue == 1:
                pret = cadouri_magazin[c]['pret']
                scor = preferinte[c][p]
                print(f"   {c:<25} (Preț: {pret} RON | Potrivire: {scor}/10)")
                subtotal += pret
                satisfactie += scor

        total_cheltuit += subtotal
        print(f"   [Subtotal: {subtotal} RON | Satisfacție: {satisfactie}]")

    print(f"\n Total cheltuit: {total_cheltuit} RON")
    print(f" Rămas din Buget: {BUGET_TOTAL - total_cheltuit} RON")

optimizare_cadouri()

Status soluție: Optimal
Fericire totală(puncte): 127.0

 Cadouri pentru TATA:
   Briceag multifunctional   (Preț: 99 RON | Potrivire: 10/10)
   Lanterna                  (Preț: 75 RON | Potrivire: 9/10)
   Termos inox               (Preț: 89 RON | Potrivire: 8/10)
   Ciocolata                 (Preț: 13.99 RON | Potrivire: 7/10)
   [Subtotal: 276.99 RON | Satisfacție: 34]

 Cadouri pentru MAMA:
   Orhidee in ghiveci        (Preț: 49.99 RON | Potrivire: 10/10)
   Set unelte gradinarit     (Preț: 119.93 RON | Potrivire: 9/10)
   Ruj                       (Preț: 42 RON | Potrivire: 5/10)
   Ciocolata                 (Preț: 13.99 RON | Potrivire: 7/10)
   [Subtotal: 225.91000000000003 RON | Satisfacție: 31]

 Cadouri pentru IUBITUL:
   Rucsac sala               (Preț: 119 RON | Potrivire: 9/10)
   Set benzi elastice        (Preț: 47.48 RON | Potrivire: 8/10)
   Termos inox               (Preț: 89 RON | Potrivire: 8/10)
   Ciocolata                 (Preț: 13.99 RON | Potrivire: 3/10)
   [Sub

**Conclusion**: the optimization was successful. The algorithm utilized 99.7% of the budget (997.36 RON), leaving only 2.64 RON unspent. It prioritized high-impact gifts but intelligently used low-cost items(chocolate) as "gap fillers" to satisfy the minimum gift requirement for everyone without breaking the bank.

Current prices for gifts:

Tent(cort): https://military-shop.ro/products/cort-2-persoane-iglu-standard-oliv-mil-tec-1373 - 249.00 RON

Multifunctional knife(briceag multifunctional): https://military-shop.ro/products/briceag-multifunctional-pocket-knife-us-army-4140 - 99.00 RON

Lanterna(lantern): https://military-shop.ro/products/lanterna-6-led-anglehead-2aa-oliv-3842?keyword=LANTERNA - 75.00 RON

Orchid in a pot(orhidee in ghiveci): https://www.lidl.ro/p/phalaenopsis-doua-tije-florale/p10050624 - 49.99 RON

Gardening tool set(set unelte gradinarit): https://www.emag.ro/set-unelte-de-gradinarit-ronyes-24-piese-negru-verde-fier-ronyes1283/pd/D0ZNZJYBM/?X-Search-Id=7e9fba8d00350ce4b227&X-Product-Id=242389286&X-Search-Page=1&X-Search-Position=14&X-Section=search&X-MB=0&X-Search-Action=view - 119.93 RON

Painted ceramic vase(vaza ceramica pictata): https://www.emag.ro/set-3-vaze-sinbinta-ceramica-de-mici-dimensiuni-simplu-si-elegant-usor-de-curatat-decor-reutilizabile-versatilitate-utilizare-larga-pentru-camera-de-zi-dormitor-camera-de-studiu-crem-wkk-224/pd/DQ87LJYBM/?ref=fam#Crem - 78.65 RON

Smartwatch fitness: https://www.emag.ro/ceas-smartwatch-idealstore-extremeprotm-collection-ecran-1-3-inch-3d-high-definition-full-touch-android-ios-monitorizare-somn-perioada-menstruala-calorii-arse-rezistent-la-apa-ip68-notificari-apeluri-/pd/D34KF9MBM/?ref=fam#Silicon-Silver - 304.15 RON

Gym backpack(rucsac sala): https://military-shop.ro/products/rucsac-assault-ultra-compact-black-6489 - 119.00 RON

Set of elastic bands(set benzi elastice): https://www.emag.ro/set-3-benzi-elastice-revity-power-bands-latex-multicolor-4-5-x-2080-x-6-4-13-22-mm-rb-pb-3-29/pd/DVGMYN3BM/ - 47.48 RON

Mascara: https://www.notino.ro/maybelline/lash-sensational-sky-high-mascara-pentru-volum-si-lungire/p-16067708/ - 47.00 RON

Eye shadow palette(paleta farduri): https://www.notino.ro/anastasia-beverly-hills/fall-romance-eye-shadow-palette-paleta-cu-farduri-de-ochi/p-16271536/ - 122.00 RON

Lipstick(ruj): https://www.notino.ro/maybelline/superstay-vinyl-ink-ruj-de-buze-lichid-de-lunga-durata/p-16131830/ - 42.00 RON

External battery(baterie externa): https://www.emag.ro/baterie-externa-10000mah-25w-samsung-gray-eb-p3400xuegeu/pd/D77SC6MBM/ - 116.99 RON

Monopoly: https://www.emag.ro/joc-monopoly-app-banking-2-6-jucatori-8-ani-lb-engleza-5010996341938/pd/DL59SR3BM/ - 185.99 RON

Stainless steel thermos(termos): https://military-shop.ro/products/termos-din-otel-inoxidabil-oliv-3577?keyword=Termos%20inox - 89.00 RON

Chocolate(ciocolata): https://www.lidl.ro/p/j-d-gross-praline-de-ciocolata-cu-umplutura/p10050648#searchTrackingMasterId=Product.10050648&searchTrackingTitle=Praline+de+ciocolat%C4%83+cu+umplutur%C4%83&searchTrackingPageSize=48&searchTrackingPage=1&searchTrackingEvent=click&searchTrackingId=Product.10050648&searchTrackingQuery=Ciocolata&searchTrackingOrigPos=1&searchTrackingPos=3&searchTrackingOrigPageSize=48&searchTrackingChannel=RO&list=search/Ciocolata - 13.99 RON
